# Safety Helmet Detection - Training Notebook
## YOLOv8m Fine-Tuning for Maximum Accuracy

**Dataset**: Safety Helmet (RoboFlow) | **Model**: YOLOv8-Medium | **Classes**: 1 (safetyhelmet)

## 1. Environment Setup

In [ ]:
!nvidia-smi
!pip install ultralytics -q

In [ ]:
from ultralytics import YOLO
import torch
import os
import yaml
import glob
import shutil
from collections import Counter
from IPython.display import Image as IPImage, display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f"VRAM: {vram / 1e9:.1f} GB")

## 2. Upload & Extract Dataset

Run the cell below, then click **Choose Files** and select `safety_helmet.zip` from your machine.

In [ ]:
import os

# Auto-detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in str(get_ipython()) if hasattr(get_ipython(), '__module__') else False

if IN_COLAB:
    from google.colab import files
    print("Upload safety_helmet.zip from your machine...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    !unzip -q -o {zip_name} -d /content/
    DATASET_DIR = "/content/Safetyhelmet-detection.v1i.yolov8"
    OUTPUT_DIR = "/content/"
elif IN_KAGGLE:
    DATASET_DIR = "/kaggle/input/safety-helmet-yolov8/Safetyhelmet-detection.v1i.yolov8"
    OUTPUT_DIR = "/kaggle/working/"
else:
    DATASET_DIR = "./Safetyhelmet-detection.v1i.yolov8"
    OUTPUT_DIR = "./"

DATA_YAML = os.path.join(OUTPUT_DIR, "data.yaml")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

# Verify dataset exists
if not os.path.exists(DATASET_DIR):
    print(f"\nERROR: Dataset not found at {DATASET_DIR}")
    print("Make sure you uploaded the zip file and it extracted correctly.")
else:
    print(f"Dataset found! Images: {len(os.listdir(os.path.join(DATASET_DIR, 'train', 'images')))} train")

## 3. Configuration

In [ ]:
# --- Model ---
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 16
PATIENCE = 20
SEED = 42

print(f"Model: {MODEL_NAME}")
print(f"Image size: {IMAGE_SIZE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")

## 4. Prepare Dataset Configuration

In [ ]:
with open(os.path.join(DATASET_DIR, "data.yaml"), 'r') as f:
    data_config = yaml.safe_load(f)

data_config['train'] = os.path.join(DATASET_DIR, "train", "images")
data_config['val'] = os.path.join(DATASET_DIR, "valid", "images")
data_config.pop('test', None)

with open(DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

## 5. Dataset Analysis

In [ ]:
class_names = {0: 'safetyhelmet'}
class_counts = Counter()
split_counts = {'train': Counter(), 'valid': Counter()}

for split in ['train', 'valid']:
    label_dir = os.path.join(DATASET_DIR, split, 'labels')
    n_images = len(glob.glob(os.path.join(label_dir, '*.txt')))
    for label_file in glob.glob(os.path.join(label_dir, '*.txt')):
        with open(label_file, 'r') as f:
            for line in f:
                cid = int(line.strip().split()[0])
                class_counts[cid] += 1
                split_counts[split][cid] += 1
    print(f"{split}: {n_images} images")

print(f"\nTotal annotations: {sum(class_counts.values())}")
for cid, count in sorted(class_counts.items()):
    print(f"  {class_names[cid]}: {count}")
print(f"\nTrain/Val ratio: {split_counts['train'][0]}:{split_counts['valid'][0]}")

## 6. Train Model

In [ ]:
model = YOLO(MODEL_NAME)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    workers=4,
    seed=SEED,

    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_bias_lr=0.01,

    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.4,

    # Strategy
    close_mosaic=10,
    amp=True,

    # Output
    project=OUTPUT_DIR,
    name="safety_helmet_training",
    exist_ok=True,
    verbose=True,
)

## 7. Training Results

In [ ]:
train_dir = os.path.join(OUTPUT_DIR, "safety_helmet_training")

plots = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'P_curve.png',
    'R_curve.png',
    'PR_curve.png',
    'F1_curve.png',
]

for name in plots:
    path = os.path.join(train_dir, name)
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"  {name}")
        print(f"{'='*60}")
        display(IPImage(filename=path, width=800))

## 8. Validation Evaluation

In [ ]:
BEST_WEIGHTS = os.path.join(OUTPUT_DIR, "safety_helmet_training", "weights", "best.pt")
best_model = YOLO(BEST_WEIGHTS)

val_results = best_model.val(
    data=DATA_YAML,
    split='val',
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.6,
    verbose=True,
)

print(f"\n{'='*40}")
print(f"  VALIDATION RESULTS")
print(f"{'='*40}")
print(f"  mAP50:      {val_results.box.map50:.4f}")
print(f"  mAP50-95:   {val_results.box.map:.4f}")
print(f"  Precision:  {val_results.box.mp:.4f}")
print(f"  Recall:     {val_results.box.mr:.4f}")

for i, name in val_results.names.items():
    print(f"  {name} AP50: {val_results.box.ap50[i]:.4f}")

## 9. Export Model

In [ ]:
output_weights = os.path.join(OUTPUT_DIR, "helmet_best.pt")
shutil.copy(BEST_WEIGHTS, output_weights)
print(f"Best weights saved to {output_weights}")
print(f"Size: {os.path.getsize(output_weights) / 1e6:.1f} MB")

# Download on Colab
if IN_COLAB:
    from google.colab import files
    files.download(output_weights)

## 10. Output Files

In [ ]:
print("Output files:")
print("-" * 60)
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        filepath = os.path.join(root, f)
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        if size_mb > 0.1:
            print(f"  {filepath} ({size_mb:.1f} MB)")